In [ ]:
# import libraries
import pandas as pd
import ee
import geemap

## Connect to GOOGLE EARTH ENGINE

In [ ]:
# Authenticate GEE
ee.Authenticate()

In [ ]:
# Initialize GEE
ee.Initialize()

print("Google Earth Engine initialized successfully!")

In [ ]:
# Center coordinates to show map
fct_center =  (9.056266, 7.498522)

## Visualise Parameters

In [ ]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 300, "max" :3000, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 300, "max" :3000, "bands": ["B8", "B4", "B3"]}


# Spectral indices visualization parameters
# NDVI
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",  
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#66bd63",
        "#1a9850",
        "#006837",  
    ],
}

# NDBI
ndbi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#f7f7f7",  
        "#fddbc7",
        "#f4a582",
        "#d6604d",
        "#b2182b",  
    ],
}

# NDWI
ndwi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#543005",  
        "#8c510a",
        "#d8b365",
        "#f6e8c3",
        "#c7eae5",
        "#5ab4ac",
        "#01665e",  
    ],
}


# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
  }

## Boundary Data

In [ ]:
# Boundary Data From FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGAs boundaries

# Create interactive map centered on Nigeria
boundary_Map = geemap.Map(center=(9.0820, 8.6753), zoom=6)

# Define styling for hollow boundaries with custom border colors
style_country = {"color": "000000", "width": 2, "fillColor": "00000000"}
style_states  = {"color": "00909F", "width": 1.5, "fillColor": "00000000"}
style_lgas    = {"color": "FF5733", "width": 0.8, "fillColor": "00000000"}

# Add styled layers
boundary_Map.addLayer(fao_gaul_l0.style(**style_country), {}, 'Country Boundaries')
boundary_Map.addLayer(fao_gaul_l1.style(**style_states), {}, 'State Boundaries')
boundary_Map.addLayer(fao_gaul_l2.style(**style_lgas), {}, 'LGA Boundaries')

boundary_Map

## Filter to Nigeria

In [ ]:
# 1. Filter FAO GAUL boundaries specifically for Nigeria
nigeria_l0 = fao_gaul_l0.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
nigeria_l1 = fao_gaul_l1.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
nigeria_l2 = fao_gaul_l2.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))

# 2. Define styled vector outlines (hollow fill so basemap shows through)
style_country = {"color": "000000", "width": 2.5, "fillColor": "00000000"}
style_states  = {"color": "00909F", "width": 1.5, "fillColor": "00000000"}
style_lgas    = {"color": "E67E22", "width": 0.8, "fillColor": "00000000"}

# 3. Initialize map centered on Nigeria
enhanced_map = geemap.Map(center=(9.0820, 8.6753), zoom=6)

# 4. Add filtered & styled boundary layers
enhanced_map.addLayer(nigeria_l0.style(**style_country), {}, 'Country Boundary')
enhanced_map.addLayer(nigeria_l1.style(**style_states), {}, 'State Boundaries')
enhanced_map.addLayer(nigeria_l2.style(**style_lgas), {}, 'LGA Boundaries')

# 5. Display the map with layer controls
enhanced_map.add_layer_control()
enhanced_map

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja"))
print(nga_l0.getInfo())

#get geometry of Abuja Boundary Feature Collection
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

#Create a map to visualize Abuja Boundary
aoi_map = geemap.Map(center=(7.0, 8.0), zoom=10)

# 7. Add layer (Fixed capital 'L' in addLayer)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'Abuja Boundary')
aoi_map


## Explore image operations
### AOI, SCL Cloud Masking, & Median Composite

In [ ]:
# Cloud masking function using SCL (Scene Classification Layer)
def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Keep clear land (4), vegetation (5), water (6), unclassified (7)
    mask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    return image.updateMask(mask)

# Download and process Sentinel-2 collection over Abuja
def get_sentinel2_composite(aoi, start_date='2022-01-01', end_date='2022-01-31'):
    s2_img_col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')   # All Sentinel-2 collection
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
    )
    return s2_img_col

# Call the function to create s2_img_col
s2_img_col = get_sentinel2_composite(aoi)

# Collection Properties
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# Get the first image in the collection and print its properties
first_s2_img = s2_img_col.first()
print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")

# Bands in S2 image
print(f"Bands in S2 image: {first_s2_img.bandNames().getInfo()}")

# Select just "B4"
red_band_s2 = first_s2_img.select('B4')
print(f"Red band (B4) in S2 image: {red_band_s2.bandNames().getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_map.addLayer(first_s2_img, vis_params_s2, 'First S2 Image')
s2_map

In [ ]:
# Mosaic the entire S2 collection & clip to FCT extent [Spatial Mosaic]
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(aoi)
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

# Create a fresh map instance for the mosaic view
s2_mosaic_map = geemap.Map(center=[9.0765, 7.3986], zoom=9)

# Add layer to the new map
s2_mosaic_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")

# Display the new map
s2_mosaic_map

## Median of the S2 collection

In [ ]:
# Median of the S2 collection
s2_img_median = s2_img_col.median()
print(f"Median of S2 collection: {s2_img_median.getInfo()}")

# Visualise s2 image
vis_params_s2 = {
    'bands': ['B8', 'B4', 'B3'],  # RGB bands
    'min': 300,
    'max': 3000,
    'gamma': 1.4
}
s2_median_map = geemap.Map(center=(7.13, 8.84), zoom=8)
s2_median_map.addLayer(s2_img_median, vis_params_s2, 'S2 Median Image')
s2_median_map

Downloading Road Data & Computing Distance to road using GEE

In [ ]:
# Loading the GRIP4 Africa Roads dataset
from folium import Map


roads_africa = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Africa")

# Extracting roads within the study area
roads_aoi = roads_africa.filterBounds(aoi_bbox)

# Converting road vectors to raster
roads_raster = ee.Image().float().paint(roads_aoi, 1).clip(aoi)


# Computing Euclidean Distance to Roads (meters)
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_roads")
    .clip(aoi)
)

# Print maximum distance value
max_distance = distance_to_roads.reduceRegion(reducer=ee.Reducer.max(),geometry=aoi,scale=1000,maxPixels=1e9)

print("Maximum Distance to Road (m):")
print(max_distance.getInfo())

# Visualization Parameters

vis_params_roads_vector = {"color": "red"}

vis_params_roads_raster = {"min": 0,"max": 1,"palette": ["white", "black"]}

vis_params_dist_road = {"min": 0,"max": 5000,"palette": ["blue","cyan","green","yellow","orange","red"]}

# Layers Display 

thematic_map_1 = aoi_map

thematic_map_1.addLayer(roads_raster,vis_params_roads_raster,"Road Raster")

thematic_map_1.addLayer(distance_to_roads.select("dist_to_roads"),vis_params_dist_road,"Distance to Road")

thematic_map_1.addLayer(roads_aoi,vis_params_roads_vector,"Road Vector")

# Display Map
thematic_map_1

Downloading Nighttime Light Data using GEE

In [ ]:

# Function to compute annual mean nighttime lights
def get_nighttime_lights(year, study_extent):
    """
    Returns annual mean VIIRS Nighttime Lights (avg_rad)
    for a given year and study area.
    """
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    composite = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
        .filterDate(start, end)
        .select("avg_rad")
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )

    return composite


# Generating Nighttime Lights Images

ntl_2015 = get_nighttime_lights(2015, aoi)
ntl_2020 = get_nighttime_lights(2020, aoi)
ntl_2025 = get_nighttime_lights(2025, aoi)

# Visualization Parameters

vis_params_ntl = {"min": 0,"max": 60,
    "palette": ["black","purple","blue","cyan","green","yellow","orange","red","white"]
}

# Creating Map

thematic_map_2 = geemap.Map(center=[9.05, 7.49], zoom=10)

# Abuja Boundary
thematic_map_2.addLayer(fct_l0, {}, "Abuja Boundary")

# Nighttime Lights Layers
thematic_map_2.addLayer(ntl_2015,vis_params_ntl,"Nighttime Lights - 2015")

thematic_map_2.addLayer(ntl_2020,vis_params_ntl,"Nighttime Lights - 2020")

thematic_map_2.addLayer(ntl_2025,vis_params_ntl,"Nighttime Lights - 2025")

# Display map
thematic_map_2

Function to export a FeatureCollection/Vector to Google Drive

In [ ]:
# Exporting FeatureCollection / Vector to Google Drive

def export_featurecollection_to_drive(
    feature_collection,
    output_filename,
    folder_name="GEE_Exports",
    file_format="SHP",
    description=None
):
    """
    Export a Google Earth Engine FeatureCollection to Google Drive.

    Parameters
    ----------
    feature_collection : ee.FeatureCollection
        FeatureCollection to export.

    output_filename : str
        Output file name.

    folder_name : str
        Google Drive folder.

    file_format : str
        Export format.
        Options: "SHP", "CSV", "GeoJSON", "KML"

    description : str
        Optional task description.
    """

    if description is None:
        description = output_filename

    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=description,
        folder=folder_name,
        fileNamePrefix=output_filename,
        fileFormat=file_format
    )

    task.start()

    print(f"Export task started successfully!")
    print(f"Description : {description}")
    print(f"Folder      : {folder_name}")
    print(f"Filename    : {output_filename}")
    print(f"Format      : {file_format}")

    return task

# Exporting Abuja AOI Boundary

export_featurecollection_to_drive(
    feature_collection=fct_l0,
    output_filename="Abuja_AOI",
    folder_name="Urban_Growth_Project",
    file_format="SHP"
)